<a href="https://colab.research.google.com/github/Aadit-5810228/Aadit-5810228-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [2]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [13]:
# Prompt: Use pandas query to give summary and head for the bikes data set.
bikes_df = tables['bikes']

print("--- Bikes Dataset Info ---")
bikes_df.info()

print("\n--- First 5 Rows ---")
display(bikes_df.head())

#Prompt: Give similar analysis for the trips data set
trips_df = tables['trips']

print("\n--- Trips Dataset Info ---")
trips_df.info()

print("\n--- First 5 Rows ---")
display(trips_df.head())

#Prompt:Give similar analysis for the trips data set
stations_df = tables['stations']

print("\n--- Stations Dataset Info ---")
stations_df.info()

print("\n--- First 5 Rows ---")
display(stations_df.head())

--- Bikes Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   bike_id        60 non-null     object
 1   bike_type      60 non-null     object
 2   purchase_year  60 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 1.5+ KB

--- First 5 Rows ---


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022



--- Trips Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   trip_id           612 non-null    object 
 1   bike_id           612 non-null    object 
 2   start_station_id  612 non-null    object 
 3   end_station_id    612 non-null    object 
 4   start_time        612 non-null    object 
 5   duration_min      582 non-null    float64
 6   user_type         612 non-null    object 
dtypes: float64(1), object(6)
memory usage: 33.6+ KB

--- First 5 Rows ---


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member



--- Stations Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_id    12 non-null     object 
 1   station_name  12 non-null     object 
 2   zone          12 non-null     object 
 3   capacity      11 non-null     float64
dtypes: float64(1), object(3)
memory usage: 516.0+ bytes

--- First 5 Rows ---


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0


In [6]:
#Prompt: Check for NA values in each of the three data sets
print("--- NA Values in Bikes Dataset ---")
print(bikes_df.isna().sum())

print("\n--- NA Values in Trips Dataset ---")
print(trips_df.isna().sum())

print("\n--- NA Values in Stations Dataset ---")
print(stations_df.isna().sum())

--- NA Values in Bikes Dataset ---
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

--- NA Values in Trips Dataset ---
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

--- NA Values in Stations Dataset ---
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64


The results suggest that in terms of null values, the Bikes clean. The NA values within the Trips data set must dropped to avoid any error during future analysis The NA within the Stations data set will be retained for now considering we need data entry for station name and zone.

In [14]:
#Prompt:Check for formatting consistency with the categorical data for each of the data sets while not considering the NA values

print("--- Unique values in bikes['bike_type'] ---")
print(bikes_df['bike_type'].value_counts(dropna=True))
print(bikes_df['purchase_year'].value_counts(dropna=True))

print("\n--- Unique values in trips['user_type'] ---")
print(trips_df['user_type'].value_counts(dropna=True))

print("\n--- Unique values in stations['zone'] ---")
print(stations_df['station_name'].value_counts(dropna=True))
print(stations_df['zone'].value_counts(dropna=True))

--- Unique values in bikes['bike_type'] ---
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64
purchase_year
2024    18
2021    16
2022    13
2023    13
Name: count, dtype: int64

--- Unique values in trips['user_type'] ---
user_type
Member     331
Casual     190
member      35
MEMBER      28
CASUAL      11
Casual       9
casual       8
Name: count, dtype: int64

--- Unique values in stations['zone'] ---
station_name
Market Square        1
Railway Station      1
Canal Basin          1
University Gate      1
Park Road            1
Library Corner       1
Hospital Entrance    1
Riverside Walk       1
Old Town Hall        1
Retail Park          1
Sports Centre        1
Cathedral Close      1
Name: count, dtype: int64
zone
Centre    5
North     2
West      2
South     2
East      1
Name: count, dtype: int64


The output suggests that the categorical data within the Bikes and Trips data sets has several entries within the same category, however the formatting is not consistent. This needs to cleaned as well.

In [11]:
# Prompt: The next check needs to be for identifying any duplicates within the unique keys across the three data sets. For the Bikes data set, this check needs to be done on the bike_id column. For the Trips data set, this check needs to be done on the trip_id column. Lastly, on the Stations data set, this check needs to be done on the station_id column

print("--- Duplicates in Bikes Dataset (bike_id) ---")
bikes_duplicates = bikes_df[bikes_df.duplicated(subset=['bike_id'], keep=False)]
print(f"Number of duplicate bike_id rows: {len(bikes_duplicates)}")
if len(bikes_duplicates) > 0:
    display(bikes_duplicates)

print("\n--- Duplicates in Trips Dataset (trip_id) ---")
trips_duplicates = trips_df[trips_df.duplicated(subset=['trip_id'], keep=False)]
print(f"Number of duplicate trip_id rows: {len(trips_duplicates)}")
if len(trips_duplicates) > 0:
    display(trips_duplicates)

print("\n--- Duplicates in Stations Dataset (station_id) ---")
stations_duplicates = stations_df[stations_df.duplicated(subset=['station_id'], keep=False)]
print(f"Number of duplicate station_id rows: {len(stations_duplicates)}")
if len(stations_duplicates) > 0:
    display(stations_duplicates)

--- Duplicates in Bikes Dataset (bike_id) ---
Number of duplicate bike_id rows: 0

--- Duplicates in Trips Dataset (trip_id) ---
Number of duplicate trip_id rows: 24


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member



--- Duplicates in Stations Dataset (station_id) ---
Number of duplicate station_id rows: 0


The output suggests that the Bikes and the Stations data set does not have any duplicates within the their primary key columns. However, the Trips data set has 24 duplicate entries. These entries need to be further checked if they are pure duplicates in terms of the data entry.

In [12]:
# Prompt: Within the duplicates in the Trips data set that were identifies using the trip_id column. Check if each duplicate entries are pure duplicates in term of the entries done in each of the columns

# Identify rows that are duplicates when considering ALL columns
pure_duplicates = trips_df[trips_df.duplicated(keep=False)]
print(f"Number of rows with duplicate entries across ALL columns: {len(pure_duplicates)}")

# Compare this to our previous trip_id duplicate count
print(f"Number of rows with duplicate trip_id: {len(trips_duplicates)}")

if len(pure_duplicates) == len(trips_duplicates):
    print("\nSuccess: All duplicate trip_id entries are 'pure' duplicates across all columns!")
else:
    print("\nWarning: There is a mismatch. Some duplicate trip_ids have differing column values.")

Number of rows with duplicate entries across ALL columns: 24
Number of rows with duplicate trip_id: 24

Success: All duplicate trip_id entries are 'pure' duplicates across all columns!


All 24 duplicates with the trips data sets are pur duplicates. Hence, the duplicates can be dropped directly during the data cleaning process

In [17]:
# Prompt: Now we need to check for any extreme outlier values. Within the Trips data set do this check within the duration_min column. Within the Stations data set, this check needs to be done within the capasity column. Use the interquartile range to check the outliers and list them.

# Helper function to find outliers using IQR
def find_iqr_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Column: {column}")
    print(f"  Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
    print(f"  Number of outliers found: {len(outliers)}")
    return outliers

print("=== Outliers in Trips Dataset ('duration_min') ===")
trips_outliers = find_iqr_outliers(trips_df, 'duration_min')
if len(trips_outliers) > 0:
    display(trips_outliers.sort_values(by='duration_min', ascending=False))

print("\n=== Outliers in Stations Dataset ('capacity') ===")
stations_outliers = find_iqr_outliers(stations_df, 'capacity')
if len(stations_outliers) > 0:
    display(stations_outliers.sort_values(by='capacity', ascending=False))

=== Outliers in Trips Dataset ('duration_min') ===
Column: duration_min
  Q1: 12.00, Q3: 22.00, IQR: 10.00
  Lower bound: -3.00, Upper bound: 37.00
  Number of outliers found: 20


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member
127,T0127,B025,S08,S09,2025-03-07 09:01:00,90.0,Member
357,T0352,B042,S03,S02,2025-03-19 08:59:00,67.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
520,T0512,B045,S07,S04,26/03/2025 18:32,51.0,Member
37,T0038,B016,S12,S09,2025-03-02 12:23:00,50.0,Casual
262,T0260,B025,S04,S04,15/03/2025 07:29,46.0,Member



=== Outliers in Stations Dataset ('capacity') ===
Column: capacity
  Q1: 19.00, Q3: 27.50, IQR: 8.50
  Lower bound: 6.25, Upper bound: 40.25
  Number of outliers found: 0


Within the Trips data set, all the entries that are above 37 min are considered as outliers. However, upon a manual check over all the outliers pointed out, I have come to the conclusion that 90 min can be set as the upper bound due the plausible nature of the duration value.

No outliers were founf in the capacity data set.

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [20]:
#Data Quality Cleaning
#Now we need to clean the data. First drop all the NA values witin just the Trips data set
trips_df.dropna(inplace=True)

#Next drop all the pure duplicates identified within the trips data set
trips_df.drop_duplicates(subset=['trip_id'], keep=False, inplace=True)

#Next drop the entries within the trips data set that have the duration_min above 90
trips_df = trips_df[trips_df['duration_min'] <= 90]

/tmp/ipykernel_2396/2372872393.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trips_df.dropna(inplace=True)
/tmp/ipykernel_2396/2372872393.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trips_df.drop_duplicates(subset=['trip_id'], keep=False, inplace=True)


In [ ]:
#Categorical Data Cleaning
#Within the

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*